# Lecture 8: CNN Architectures

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 8, with material from Stanford CS231n 2026, Lecture 6.*

This lecture is a tour of the networks that won ImageNet, and the design ideas behind each one. The best way to understand them is to **count**: how much memory, how many parameters, how much computation each layer needs.

**What you'll learn**
1. How to compute a layer's output size, parameters, and computation.
2. **AlexNet**: where its memory, parameters, and computation live.
3. **VGG**: simple design rules, and why 3×3 filters.
4. **GoogLeNet**: tricks for efficiency.
5. **ResNet**: why very deep networks failed, and how residual connections fixed it.
6. Efficient networks since then, and which architecture to use.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

## 1. ImageNet winners

| Year | Network | Layers | Top-5 error |
|---|---|---|---|
| 2011 | (hand-designed features) | — | 25.8% |
| 2012 | **AlexNet** | 8 | 16.4% |
| 2013 | ZFNet | 8 | 11.7% |
| 2014 | **VGG** (2nd place) | 19 | 7.3% |
| 2014 | **GoogLeNet** | 22 | 6.7% |
| 2015 | **ResNet** | 152 | 3.6% |
| 2017 | SENet | 152 | 2.3% |

From 8 layers to 152 in three years: "the revolution of depth".

## 2. Counting a layer's cost

For a **convolution** with $C_{out}$ filters of size $K\times K$ on an input of $C_{in}\times H\times W$:
* **output size:** $H' = (H - K + 2P)/S + 1$
* **parameters:** $C_{out}\times(C_{in}\cdot K\cdot K + 1)$ (weights + one bias per filter)
* **computation** (multiply-adds): (number of outputs) × (work per output) $= C_{out}\cdot H'\cdot W' \times C_{in}\cdot K\cdot K$
* **memory** for the output: $C_{out}\cdot H'\cdot W'$ numbers × 4 bytes

For a **fully connected** layer from $n$ to $m$ numbers: $m\times(n + 1)$ parameters and $n\times m$ multiply-adds. **Pooling** has no parameters.

**By hand: AlexNet's first layer.** Input 3×227×227; 64 filters of 11×11, stride 4, pad 2:

In [ ]:
c_in, size_in = 3, 227
c_out, k, stride, pad = 64, 11, 4, 2

size_out = (size_in - k + 2 * pad) // stride + 1
params = c_out * (c_in * k * k + 1)
multiply_adds = c_out * size_out * size_out * c_in * k * k
memory_kb = c_out * size_out * size_out * 4 / 1024

print(f"output: {c_out} x {size_out} x {size_out}")
print(f"parameters: {params:,}")
print(f"multiply-adds: {multiply_adds / 1e6:.1f} million")
print(f"memory for the output: {memory_kb:.0f} KB")

Let's turn this into a small calculator that walks through a whole network. Each layer is described by a short tuple.

In [ ]:
def analyze(network, channels, size, show=True):
    # network: list of ("conv", filters, kernel, stride, pad), ("pool", kernel, stride), or ("fc", outputs)
    total_params, total_flops, total_memory = 0, 0, 0
    if show:
        print(f"{'layer':22s} {'output':>16s} {'params':>12s} {'mult-adds':>12s}")
    for layer in network:
        if layer[0] == "conv":
            _, filters, k, s, p = layer
            new_size = (size - k + 2 * p) // s + 1
            params = filters * (channels * k * k + 1)
            flops = filters * new_size * new_size * channels * k * k
            channels, size = filters, new_size
            name, shape = f"conv {k}x{k}, {filters}", f"{channels}x{size}x{size}"
        elif layer[0] == "pool":
            _, k, s = layer
            size = (size - k) // s + 1
            params, flops = 0, 0
            name, shape = f"pool {k}x{k}", f"{channels}x{size}x{size}"
        else:                                                       # fully connected
            _, outputs = layer
            inputs = channels * size * size
            params, flops = outputs * (inputs + 1), inputs * outputs
            channels, size = outputs, 1
            name, shape = f"fc {outputs}", f"{outputs}"
        memory = channels * size * size * 4 / 1024                  # KB for this layer's output
        total_params += params; total_flops += flops; total_memory += memory
        if show:
            print(f"{name:22s} {shape:>16s} {params:12,d} {flops:12,d}")
    if show:
        print(f"{'TOTAL':22s} {'':>16s} {total_params:12,d} {total_flops:12,d}")
    return total_params, total_flops, total_memory

## 3. AlexNet (2012)

5 convolutional layers, max pooling, 3 fully connected layers, ReLU. Trained on two GPUs with only 3 GB of memory each.

In [ ]:
alexnet = [("conv", 64, 11, 4, 2), ("pool", 3, 2),
           ("conv", 192, 5, 1, 2), ("pool", 3, 2),
           ("conv", 384, 3, 1, 1), ("conv", 256, 3, 1, 1), ("conv", 256, 3, 1, 1), ("pool", 3, 2),
           ("fc", 4096), ("fc", 4096), ("fc", 1000)]
alex_params, alex_flops, alex_memory = analyze(alexnet, channels=3, size=227)

**Three trends to notice (true for most CNNs):**
* **Most of the memory** is in the early conv layers (big images).
* **Nearly all the parameters** are in the fully connected layers (look at the 37 million in the first one!).
* **Most of the computation** is in the conv layers.

How were these layer sizes chosen? Trial and error. **ZFNet** (2013) was basically a bigger AlexNet, found the same way.

## 4. VGG (2014): simple design rules

VGG replaced trial and error with three rules:
* **All convolutions are 3×3, stride 1, pad 1.**
* **All max pools are 2×2, stride 2.**
* **After each pool, double the number of channels.**

With 5 stages of convolutions and 3 FC layers, that gives **VGG-16**:

In [ ]:
vgg16 = []
for channels, n_convs in [(64, 2), (128, 2), (256, 3), (512, 3), (512, 3)]:
    for _ in range(n_convs):
        vgg16.append(("conv", channels, 3, 1, 1))
    vgg16.append(("pool", 2, 2))
vgg16 += [("fc", 4096), ("fc", 4096), ("fc", 1000)]

vgg_params, vgg_flops, vgg_memory = analyze(vgg16, channels=3, size=224, show=False)
print(f"VGG-16: {vgg_params:,} parameters  (the official number is 138,357,544)")
print(f"VGG-16 vs. AlexNet: {vgg_params / alex_params:.1f}x the parameters, "
      f"{vgg_flops / alex_flops:.0f}x the computation, {vgg_memory / alex_memory:.0f}x the memory")

### Why 3×3 filters?
Two 3×3 convolutions in a row have the **same receptive field** as one 5×5 (Lecture 7: $1 + 2 + 2 = 5$), and three 3×3 equal one 7×7. But they need **fewer parameters**, and have **more ReLUs** in between (more nonlinearity). With $C$ channels in and out:

In [ ]:
C = 256
print(f"one 5x5 conv:   {25 * C * C:,} weights  (25 C^2)")
print(f"two 3x3 convs:  {2 * 9 * C * C:,} weights  (18 C^2)")
print(f"one 7x7 conv:   {49 * C * C:,} weights  (49 C^2)")
print(f"three 3x3 convs: {3 * 9 * C * C:,} weights  (27 C^2)")

### Why double the channels after each pool?
Pooling halves the height and width (4× fewer positions), and doubling the channels makes each conv 4× more expensive per position. The two cancel, so **every stage costs the same amount of computation**:

In [ ]:
for stage, (channels, size) in enumerate([(64, 224), (128, 112), (256, 56), (512, 28)], start=1):
    flops = channels * size * size * channels * 3 * 3       # one 3x3 conv, channels -> channels
    print(f"stage {stage}: 3x3 conv with {channels:3d} channels at {size:3d}x{size:3d}: {flops / 1e9:.2f} billion multiply-adds")

## 5. GoogLeNet (2014): focus on efficiency

GoogLeNet won with **far less computation** than VGG, using three tricks.

**Trick 1: a "stem" that shrinks the image quickly.** VGG spends most of its computation on the big early layers. GoogLeNet uses a few cheap layers to go from 224×224 to 28×28 right away.

**Trick 2: the Inception module.** Run several filter sizes in parallel (1×1, 3×3, 5×5, and pooling) and stack the results. To keep that affordable, first use **1×1 convolutions to reduce the number of channels** (a "bottleneck"). For the first module (input 192×28×28):

In [ ]:
def conv_cost(c_in, c_out, k, size=28):
    # multiply-adds for a 'same'-padded conv on a size x size input
    return c_out * size * size * c_in * k * k

naive = conv_cost(192, 64, 1) + conv_cost(192, 128, 3) + conv_cost(192, 32, 5)
with_bottlenecks = (conv_cost(192, 64, 1)
                    + conv_cost(192, 96, 1) + conv_cost(96, 128, 3)      # 1x1 down to 96, then 3x3
                    + conv_cost(192, 16, 1) + conv_cost(16, 32, 5)       # 1x1 down to 16, then 5x5
                    + conv_cost(192, 32, 1))                             # pooling branch's 1x1
print(f"without bottlenecks: {naive / 1e6:.0f} million multiply-adds")
print(f"with 1x1 bottlenecks: {with_bottlenecks / 1e6:.0f} million multiply-adds")

**Trick 3: no big fully connected layers.** Instead, **global average pooling** averages each channel's 7×7 map down to one number (1024×7×7 → 1024), followed by one small linear layer. Remember that most of AlexNet's and VGG's parameters were in the FC layers:

In [ ]:
print(f"flatten 1024x7x7, then fc to 1000:   {1024 * 7 * 7 * 1000:,} weights")
print(f"global average pool, then fc to 1000: {1024 * 1000:,} weights")

(GoogLeNet also added extra "auxiliary" classifiers in the middle of the network, because gradients didn't reach the early layers well. Once **batch normalization** came along, that trick was no longer needed.)

## 6. ResNet (2015): going much deeper

**With batch normalization, we can train networks with 10+ layers. What happens if we go much deeper?** Surprisingly, a 56-layer network did **worse** than a 20-layer one, even on the **training** data.

So it's not overfitting (that would mean *good* training accuracy). It's an **optimization problem**: a deeper network *could* copy the shallow one and set its extra layers to "do nothing" (the identity), but gradient descent doesn't find that solution.

**The fix: make "do nothing" easy.** A **residual block** adds its input back to its output:

$$\text{output} = F(x) + x$$

If the block's weights are zero, $F(x) = 0$ and the block passes $x$ through unchanged. The layers only have to learn the *change* to $x$.

### Let's reproduce the problem and the fix
We'll train fully connected networks on our digits: a shallow plain one, a deep plain one, and a deep residual one. Each **block** is two linear layers:
* **plain block:** $h \leftarrow \text{ReLU}(W_2\,\text{ReLU}(W_1h))$
* **residual block:** $h \leftarrow \text{ReLU}(h + W_2\,\text{ReLU}(W_1h))$

In [ ]:
digits = load_digits()
X = digits.data / 16.0 - 0.3
y = digits.target

def softmax_loss(scores, labels):
    shifted = scores - scores.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(probs[np.arange(len(labels)), labels] + 1e-12))
    probs[np.arange(len(labels)), labels] -= 1
    return loss, probs / len(labels)

In [ ]:
def make_network(n_blocks, residual, width=32, seed=0):
    rng = np.random.default_rng(seed)
    scale = np.sqrt(2 / width)
    net = {"residual": residual,
           "W_in": rng.normal(0, np.sqrt(2 / 64), size=(64, width)),
           "W_out": rng.normal(0, scale, size=(width, 10)),
           "W1": [rng.normal(0, scale, size=(width, width)) for _ in range(n_blocks)],
           "W2": [rng.normal(0, scale, size=(width, width)) for _ in range(n_blocks)]}
    if residual:
        net["W2"] = [0.1 * W for W in net["W2"]]     # start each block close to "do nothing"
    return net

The training step: forward through the blocks (remembering what the backward pass needs), then backward in reverse. **The one difference for residual blocks** is in the backward pass: the shortcut is an add node (Lecture 6: "add distributes"), so the gradient also flows **straight through** to the block's input.

In [ ]:
def train_step(net, X_batch, y_batch, lr=0.02):
    # ---- forward ----
    h = np.maximum(0, X_batch @ net["W_in"])
    saved = []
    for W1, W2 in zip(net["W1"], net["W2"]):
        z1 = h @ W1
        a1 = np.maximum(0, z1)
        z2 = a1 @ W2
        before_relu = h + z2 if net["residual"] else z2
        saved.append((h, z1, a1, before_relu))
        h = np.maximum(0, before_relu)
    loss, d_scores = softmax_loss(h @ net["W_out"], y_batch)
    # ---- backward ----
    grad_W_out = h.T @ d_scores
    d_h = d_scores @ net["W_out"].T
    for i in reversed(range(len(net["W1"]))):
        h_in, z1, a1, before_relu = saved[i]
        d_before = d_h * (before_relu > 0)                  # back through the block's final ReLU
        grad_W2 = a1.T @ d_before
        d_z1 = (d_before @ net["W2"][i].T) * (z1 > 0)
        grad_W1 = h_in.T @ d_z1
        d_h = d_z1 @ net["W1"][i].T
        if net["residual"]:
            d_h = d_h + d_before                            # the shortcut: gradient flows straight through
        net["W1"][i] -= lr * grad_W1
        net["W2"][i] -= lr * grad_W2
    d_input = d_h * (X_batch @ net["W_in"] > 0)
    net["W_in"] -= lr * X_batch.T @ d_input
    net["W_out"] -= lr * grad_W_out
    return loss

In [ ]:
def final_training_loss(n_blocks, residual, steps=1500, seed=0):
    net = make_network(n_blocks, residual, seed=seed)
    rng = np.random.default_rng(seed)
    losses = []
    for step in range(steps):
        batch = rng.integers(0, 1300, 64)
        with np.errstate(all="ignore"):
            losses.append(train_step(net, X[batch], y[batch]))
    return np.mean(losses[-100:])                          # average over the last 100 steps

print(f"{'network':26s} {'layers':>6s} {'final training loss':>20s}")
for n_blocks, residual in [(2, False), (25, False), (25, True)]:
    name = ("residual" if residual else "plain") + f", {n_blocks} blocks"
    print(f"{name:26s} {2 * n_blocks + 2:6d} {final_training_loss(n_blocks, residual):20.4f}")

**What we see:** the 52-layer plain network can't even fit its **training** data (its loss is still high, about 1.9, not far below the 2.3 it started at), while the shallow one fits easily. The same 52 layers **with shortcuts** train perfectly. That's the ResNet paper's discovery in miniature.

### The full ResNet design
* A stack of residual blocks, each with **two 3×3 convs** (like VGG).
* **Stages**: the first block of each stage halves the resolution and doubles the channels.
* GoogLeNet's quick **stem** at the start, and **global average pooling** + one linear layer at the end.
* Depths of 18, 34, 50, 101, 152 layers. ResNet-152 won ImageNet 2015 and is **still cheaper than VGG-16**.

**Bottleneck blocks** (ResNet-50 and deeper): 1×1 conv down to $C$ channels, 3×3 conv, 1×1 conv back up to $4C$. More layers, but less computation than two 3×3 convs:

In [ ]:
C, size = 64, 56
basic_block = 2 * (C * size * size * C * 9)                                  # two 3x3 convs on C channels
bottleneck_block = (C * size * size * 4 * C) + (C * size * size * C * 9) + (4 * C * size * size * C)
print(f"basic block:      {basic_block / (size * size * C * C):.0f} x H W C^2 multiply-adds")
print(f"bottleneck block: {bottleneck_block / (size * size * C * C):.0f} x H W C^2 multiply-adds")

## 7. After ResNet: efficient networks

Once very deep networks were possible, the focus moved to **getting more accuracy for less computation**:
* **ResNeXt:** split each bottleneck into many parallel thin paths ("grouped convolutions"): better accuracy for the same cost.
* **SENet** (2017 winner): add a tiny branch that learns to re-weight each channel.
* **MobileNet:** replace each 3×3 conv with a **depthwise** 3×3 conv (each channel filtered on its own) followed by a **1×1** conv that mixes channels. Much cheaper:

In [ ]:
for C in [64, 256]:
    standard = 9 * C * C                # per pixel: 3x3 conv, C -> C
    separable = 9 * C + C * C           # depthwise 3x3, then 1x1
    print(f"C = {C}: standard {standard:,} vs depthwise-separable {separable:,}  ({standard / separable:.1f}x cheaper)")

* **Neural Architecture Search (NAS):** let an algorithm design the network (very expensive at first, but it found efficient designs).
* Since 2020: **EfficientNet** (scale depth, width, and resolution together), **Vision Transformers** (Lecture 13), and **ConvNeXt** (a ResNet modernized with Transformer-era tricks).

## 8. Comparing them, and which one to use

Using torchvision's reported ImageNet top-1 accuracies and our computed costs:

In [ ]:
models = {  # name: (top-1 accuracy %, billions of multiply-adds, millions of parameters)
    "AlexNet": (56.5, alex_flops / 1e9, alex_params / 1e6),
    "VGG-16": (71.6, vgg_flops / 1e9, vgg_params / 1e6),
    "GoogLeNet": (69.8, 1.5, 6.6),
    "ResNet-18": (69.8, 1.8, 11.7),
    "ResNet-50": (76.1, 4.1, 25.6),
    "ResNet-152": (78.3, 11.5, 60.2),
    "MobileNetV2": (71.9, 0.3, 3.5),
}
plt.figure(figsize=(7, 4.5))
for name, (acc, gflops, params) in models.items():
    plt.scatter(gflops, acc, s=params * 3, alpha=0.5)
    plt.annotate(name, (gflops, acc), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xscale("log"); plt.xlabel("billions of multiply-adds (log scale)"); plt.ylabel("ImageNet top-1 accuracy (%)")
plt.title("accuracy vs. computation (bubble size = parameters)")
plt.show()

**Which architecture should you use? "Don't be a hero."** For most problems, use an existing architecture, and start from **pretrained** weights (transfer learning, Lecture 11):
* if you just want accuracy: **ResNet-50 or ResNet-101** are great choices;
* if you need speed (phones, real time): **MobileNet** or **ShuffleNet**.

## 9. In PyTorch (optional, run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch torchvision. weights=None builds the network without downloading anything.
import torchvision
for name in ["alexnet", "vgg16", "resnet18", "resnet50", "mobilenet_v2"]:
    model = getattr(torchvision.models, name)(weights=None)
    print(f"{name:13s}: {sum(p.numel() for p in model.parameters()):>12,d} parameters")

## Summary
| Network | Key idea |
|---|---|
| AlexNet | Deep CNN + GPUs; most parameters in the FC layers |
| VGG | Only 3×3 convs and 2×2 pools; double channels after each pool |
| GoogLeNet | Efficiency: quick stem, 1×1 bottlenecks, global average pooling |
| ResNet | Residual blocks $F(x) + x$ make very deep networks trainable |
| MobileNet & co. | Accuracy per unit of computation (depthwise-separable convs, NAS) |

### Check your understanding
1. Where are most of AlexNet's parameters, and where is most of its computation?
2. Why does VGG use 3×3 filters?
3. A 56-layer network has higher *training* error than a 20-layer one. Is it overfitting?
4. Why does a residual block make it easy to learn the identity?

<details><summary>Answers</summary>

1. Parameters: the fully connected layers. Computation: the conv layers.
2. Stacked 3×3s cover the same area as one big filter, with fewer parameters and more nonlinearities.
3. No: overfitting would mean low training error. It's an optimization problem.
4. With zero weights, $F(x) = 0$ and the output is just $x$.
</details>

**Next:** Lecture 9, hardware and software.